#### Import the library

In [44]:
# NOTE: json = read JSON
# NOTE: Path = safer file paths than raw strings
# NOTE: pandas = convert to table + export CSV

import json
from pathlib import Path
import pandas as pd

#### File path + existence check

In [45]:
# NOTE: Put the JSON file in the same folder as your notebook,
# or change the path here.

input_path = Path("medium_orders_practice_v2.json")
input_path

WindowsPath('medium_orders_practice_v2.json')

In [46]:
# NOTE: Always check the file exists before reading, saves time.

print("Exists? ", input_path.exists())
print("Absoluate Path", input_path.resolve())

Exists?  True
Absoluate Path C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\json-practice-medium\medium_orders_practice_v2.json


#### Load JSON + inspect the "shape"


In [47]:
# NOTE: Read the file text then parse JSON into Python objects (dict/list).
first_read = json.loads(input_path.read_text(encoding="utf-8"))
first_read

{'meta': {'source': 'demo_api_medium_v2',
  'pulled_at': '2026-02-06T03:45:00Z',
  'notes': 'Practice dataset: nested customer + shipping + items[]; includes nulls, type issues, bad timestamp, empty items'},
 'data': [{'order_id': 'ord_201',
   'order_time': '2026-02-06T00:10:15Z',
   'customer': {'customer_id': '401',
    'email': 'maya@example.com',
    'country': 'au'},
   'shipping': {'city': 'Sydney', 'state': 'NSW'},
   'items': [{'sku': 'A1', 'name': 'Notebook', 'qty': 2, 'unit_price': '4.50'},
    {'sku': 'B2', 'name': 'Pen', 'qty': '3', 'unit_price': 1.2}],
   'payment': {'method': 'card', 'currency': 'aud'},
   'discount_code': None},
  {'order_id': 'ord_202',
   'order_time': 'bad_timestamp',
   'customer': {'customer_id': 402,
    'email': 'liam@example.com',
    'country': None},
   'shipping': {'city': None, 'state': 'VIC'},
   'items': [{'sku': 'A1',
     'name': 'Notebook',
     'qty': '1',
     'unit_price': '4.50'}],
   'payment': {'method': 'paypal', 'currency': None

In [48]:
# NOTE: Confirm top level structures
print("Top-level keys: ", first_read.keys())

# NOTE: double check the Meta keys, most of the time is API
print("Meta: ", first_read.get("meta"))

# NOTE: As we can seen data is what we are looking for.
order = first_read.get("data")
print("our order type ", type(order))
print("length of our order", len(order))


Top-level keys:  dict_keys(['meta', 'data'])
Meta:  {'source': 'demo_api_medium_v2', 'pulled_at': '2026-02-06T03:45:00Z', 'notes': 'Practice dataset: nested customer + shipping + items[]; includes nulls, type issues, bad timestamp, empty items'}
our order type  <class 'list'>
length of our order 4


In [49]:
print("Let see first record of our data: ")
order[1]

Let see first record of our data: 


{'order_id': 'ord_202',
 'order_time': 'bad_timestamp',
 'customer': {'customer_id': 402,
  'email': 'liam@example.com',
  'country': None},
 'shipping': {'city': None, 'state': 'VIC'},
 'items': [{'sku': 'A1',
   'name': 'Notebook',
   'qty': '1',
   'unit_price': '4.50'}],
 'payment': {'method': 'paypal', 'currency': None},
 'discount_code': 'WELCOME10'}

#### Based on our data, we can see items is a list so we will have more than 1 row. We will create two table:
- order_df
- item_df

In [50]:
# =========================
# CELL 2 — Build "orders" table (1 row per order) by FLATTENING nested JSON
# =========================

order_rows = []

for r in order:

    c = r.get("customer")
    s = r.get("shipping")
    p = r.get("payment")

    # NOTE: orders table = 1 row per order (NO items columns here)
    order_rows.append({
        "order_id" : r.get("order_id"),
        "order_time" : r.get("order_time"),
        "customer_id" : c.get("customer_id"),
        "email": c.get("email"),
        "country": c.get("country"),
        "ship_city" : s.get("city"),
        "ship_state": s.get("state"),
        "payment_method": p.get("method"),
        "currency" : p.get("currency"),
        "discount_code": r.get("discount_code")
    })

order_df = pd.DataFrame(order_rows)
order_df

,order_id,order_time,customer_id,email,country,ship_city,ship_state,payment_method,currency,discount_code
0,ord_201,2026-02-06T00:10:15Z,401,maya@example.com,au,Sydney,NSW,card,aud,NaN
1,ord_202,bad_timestamp,402,liam@example.com,NaN,NaN,VIC,paypal,NaN,WELCOME10
2,ord_203,2026-02-06T01:05:00Z,None,no-id@example.com,AU,Brisbane,QLD,NaN,AUD,
3,ord_204,2026-02-06T02:20:30Z,404,NaN,NZ,Auckland,NaN,card,NZD,NaN


#### clean order df 

In [51]:
order_df["order_time"] = pd.to_datetime(order_df["order_time"], utc = True, errors = "coerce")
order_df["customer_id"] = order_df["customer_id"].fillna(-1).astype("int64")
order_df["country"] = order_df["country"].fillna("UNKNOWN").astype(str).str.upper()
order_df["ship_city"] = order_df["ship_city"].fillna("UNKNOWN").astype(str).str.upper()
order_df["ship_state"] = order_df["ship_state"].fillna("UNKNOWN").astype(str).str.upper()
order_df["payment_method"] = order_df["payment_method"].fillna("UNKNOWN").astype(str).str.upper()
order_df["currency"] = order_df["currency"].fillna("AUD").astype(str).str.upper()
order_df["discount_code"] = order_df["discount_code"].fillna("").astype(str).str.upper()

order_df

,order_id,order_time,customer_id,email,country,ship_city,ship_state,payment_method,currency,discount_code
0,ord_201,2026-02-06 00:10:15+00:00,401,maya@example.com,AU,SYDNEY,NSW,CARD,AUD,
1,ord_202,NaT,402,liam@example.com,UNKNOWN,UNKNOWN,VIC,PAYPAL,AUD,WELCOME10
2,ord_203,2026-02-06 01:05:00+00:00,-1,no-id@example.com,AU,BRISBANE,QLD,UNKNOWN,AUD,
3,ord_204,2026-02-06 02:20:30+00:00,404,NaN,NZ,AUCKLAND,UNKNOWN,CARD,NZD,


#### working and converting to items_df

In [52]:
order[1]

{'order_id': 'ord_202',
 'order_time': 'bad_timestamp',
 'customer': {'customer_id': 402,
  'email': 'liam@example.com',
  'country': None},
 'shipping': {'city': None, 'state': 'VIC'},
 'items': [{'sku': 'A1',
   'name': 'Notebook',
   'qty': '1',
   'unit_price': '4.50'}],
 'payment': {'method': 'paypal', 'currency': None},
 'discount_code': 'WELCOME10'}

In [53]:
items_rows = []

for r in order:
    order_id = r.get("order_id")
    order_time = r.get("order_time")
    discount_code = r.get("discount_code")
    customer_id = (r.get("customer")).get("customer_id")
    ship_city = (r.get("shipping")).get("city")
    currency = (r.get("payment")).get("currency")

    # NOTE: items is a LIST; is missing or empty -) []
    it = r.get("items") or {}

    # NOTE: explode: create 1 row for each item in the list
    for i in it:
        items_rows.append({
            "order_id": order_id,
            "order_time": order_time,
            "customer_id": customer_id,
            "ship_city": ship_city,
            "sku": i.get("sku"),
            "name": i.get("name"),
            "qty": i.get("qty"),
            "unit_price": i.get("unit_price"),
            "currency": currency,
            "discount_code": discount_code
        })

items_df = pd.DataFrame(items_rows)
items_df

,order_id,order_time,customer_id,ship_city,sku,name,qty,unit_price,currency,discount_code
0,ord_201,2026-02-06T00:10:15Z,401,Sydney,A1,Notebook,2,4.50,aud,NaN
1,ord_201,2026-02-06T00:10:15Z,401,Sydney,B2,Pen,3,1.2,aud,NaN
2,ord_202,bad_timestamp,402,NaN,A1,Notebook,1,4.50,NaN,WELCOME10
3,ord_204,2026-02-06T02:20:30Z,404,Auckland,C3,Marker,1,2.75,NZD,NaN
4,ord_204,2026-02-06T02:20:30Z,404,Auckland,D4,Eraser,2,None,NZD,NaN


#### Clean items df

In [54]:
items_df["order_time"] = pd.to_datetime(items_df["order_time"], utc= True, errors = "coerce")
items_df["customer_id"] = items_df["customer_id"].fillna(-1).astype("int64")
items_df["ship_city"] = items_df["ship_city"].fillna("UNKNOWN").astype(str).str.upper()
items_df["sku"] = items_df["sku"].fillna("").astype(str).str.upper()
items_df["qty"] = items_df["qty"].fillna("0").astype("int64")
items_df["name"] = items_df["name"].fillna("").astype(str).str.title()
items_df["unit_price"] = pd.to_numeric(items_df["unit_price"]).fillna("0")
items_df["currency"] = items_df["currency"].fillna("AUD").astype(str).str.upper()
items_df["discount_code"] = items_df["discount_code"].fillna("").astype(str).str.upper()
items_df["Line_total"] = pd.to_numeric(items_df["qty"] * items_df["unit_price"]).round(2)

items_df

,order_id,order_time,customer_id,ship_city,sku,name,qty,unit_price,currency,discount_code,Line_total
0,ord_201,2026-02-06 00:10:15+00:00,401,SYDNEY,A1,Notebook,2,4.5,AUD,,9.00
1,ord_201,2026-02-06 00:10:15+00:00,401,SYDNEY,B2,Pen,3,1.2,AUD,,3.60
2,ord_202,NaT,402,UNKNOWN,A1,Notebook,1,4.5,AUD,WELCOME10,4.50
3,ord_204,2026-02-06 02:20:30+00:00,404,AUCKLAND,C3,Marker,1,2.75,NZD,,2.75
4,ord_204,2026-02-06 02:20:30+00:00,404,AUCKLAND,D4,Eraser,2,0,NZD,,0.00


#### Export to csv

In [55]:
output_path_order = Path("order_clean.csv")
order_df.to_csv(output_path_order, index=False, encoding="utf-8")

output_path_item = Path("order_items_clean.csv")
items_df.to_csv(output_path_item, index=False, encoding="utf-8")


print("Saved CSV: ", output_path_order.resolve())
print("Saved CSV: ", output_path_item.resolve())

Saved CSV:  C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\json-practice-medium\order_clean.csv
Saved CSV:  C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\json-practice-medium\order_items_clean.csv
